# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tannusaini2110-spec/Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [ ]:
print("One row = one content page (content_hash_id), for one client, measured on one day.")
print("Table(s) used: fact_content_daily_performance (the daily grain table).")
print("Time window: a single mid-panel month, month=2026-03, to avoid the sealed final test month.")
print("Target/proxy: is_declining -- whether a page's recent 30-day impressions dropped")
print("vs. the prior 30-day window, built from this table's daily rows.")
print("Deliberately excluded: query-level data (fact_content_query_90d) -- too granular")
print("for this first pass; I'm working at the page-day level only.")

One row = one content page (content_hash_id), for one client, measured on one day.
Table(s) used: fact_content_daily_performance (the daily grain table).
Time window: a single mid-panel month, month=2026-03, to avoid the sealed final test month.
Target/proxy: is_declining -- whether a page's recent 30-day impressions dropped
vs. the prior 30-day window, built from this table's daily rows.
Deliberately excluded: query-level data (fact_content_query_90d) -- too granular
for this first pass; I'm working at the page-day level only.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [ ]:
print("FEATURES (used to predict, known before the outcome):")
print("  - gsc_avg_position (search ranking position)")
print("  - gsc_impressions (visibility)")
print("  - gsc_clicks (traffic)")
print("  - content_age_days (how old the page is)")
print("  - word_count (content length)")
print()
print("LABEL (what I'm predicting):")
print("  - is_declining -- derived from comparing recent vs prior impressions")
print()
print("CONTEXT (identifiers, not predictive features):")
print("  - client_hash_id (which client)")
print("  - content_hash_id (which page)")
print("  - report_date (which day)")
print()
print("EXCLUDED (deliberately not used):")
print("  - trend_pct / trend_direction if pre-computed in the table --")
print("    these are leakage risks since they may already encode the outcome")
print("    I'm trying to predict, not a true independent feature.")

FEATURES (used to predict, known before the outcome):
  - gsc_avg_position (search ranking position)
  - gsc_impressions (visibility)
  - gsc_clicks (traffic)
  - content_age_days (how old the page is)
  - word_count (content length)

LABEL (what I'm predicting):
  - is_declining -- derived from comparing recent vs prior impressions

CONTEXT (identifiers, not predictive features):
  - client_hash_id (which client)
  - content_hash_id (which page)
  - report_date (which day)

EXCLUDED (deliberately not used):
  - trend_pct / trend_direction if pre-computed in the table --
    these are leakage risks since they may already encode the outcome
    I'm trying to predict, not a true independent feature.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [3]:
%pip -q install duckdb huggingface_hub

import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
TABLE = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"

# Query 1: grain check -- is one row really one page-day?
grain_check = con.sql(f"""
    SELECT content_hash_id, report_date, COUNT(*) as n
    FROM {TABLE}
    GROUP BY 1, 2
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
print("Query 1 -- Grain check (should be empty):")
print(grain_check)
print()

# Query 2: row count and date span for this slice
span = con.sql(f"""
    SELECT COUNT(*) as total_rows, MIN(report_date) as earliest,
           MAX(report_date) as latest, COUNT(DISTINCT content_hash_id) as unique_pages
    FROM {TABLE}
""").df()
print("Query 2 -- Row count and date span:")
print(span)
print()

# Query 3: availability -- filter with IS TRUE
available = con.sql(f"""
    SELECT COUNT(*) as rows_available
    FROM {TABLE}
    WHERE gsc_data_available IS TRUE
""").df()
print("Query 3 -- Rows with GSC data actually available:")
print(available)

# Build the feature frame: one row per content_hash_id, aggregated over March
feat = con.sql(f"""
    SELECT content_hash_id,
           AVG(gsc_avg_position) as avg_position,
           SUM(gsc_impressions) as total_impressions,
           SUM(gsc_clicks) as total_clicks,
           AVG(ga4_engaged_sessions) as avg_engaged_sessions,
           COUNT(*) as days_seen
    FROM {TABLE}
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id
""").df()

print(f"Feature frame: {len(feat):,} pages")
print(feat.head())
print()

print("Feature availability -- knowable at the decision moment because...")
print("1. avg_position       -- known from that day's GSC data, before any future outcome.")
print("2. total_impressions  -- accumulated from past search visibility, not future.")
print("3. total_clicks       -- past traffic, observed as it happens each day.")
print("4. avg_engaged_sessions -- past GA4 engagement, recorded daily, no future info.")
print("5. days_seen          -- simply counts days with data in the window, no leakage risk.")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Query 1 -- Grain check (should be empty):
Empty DataFrame
Columns: [content_hash_id, report_date, n]
Index: []

Query 2 -- Row count and date span:
   total_rows   earliest     latest  unique_pages
0     9841378 2026-03-01 2026-03-31        331437



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Query 3 -- Rows with GSC data actually available:
   rows_available
0         3611061


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame: 176,738 pages
            content_hash_id  avg_position  total_impressions  total_clicks  \
0  content_2e6360ad20fd7107      5.145765              899.0           1.0   
1  content_ac8663da7484669a      4.909314               34.0           0.0   
2  content_d49a012dcb924e31      5.177774              329.0           0.0   
3  content_614baf2af4330bd7      4.685335              772.0           1.0   
4  content_4a1ca0fa5c177e0c      4.266667               14.0           0.0   

   avg_engaged_sessions  days_seen  
0                   NaN         31  
1                   NaN         17  
2                   NaN         31  
3                   NaN         31  
4                   NaN         10  

Feature availability -- knowable at the decision moment because...
1. avg_position       -- known from that day's GSC data, before any future outcome.
2. total_impressions  -- accumulated from past search visibility, not future.
3. total_clicks       -- past traffic, observed as

In [4]:
from sklearn.tree import DecisionTreeClassifier

# Build a simple label: bottom 30% by total_clicks = "declining" (low performing)
feat["is_declining"] = (feat["total_clicks"] <= feat["total_clicks"].quantile(0.30)).astype(int)

honest_features = ["avg_position", "total_impressions", "avg_engaged_sessions", "days_seen"]
X_honest = feat[honest_features].fillna(0)
y = feat["is_declining"]

tree_honest = DecisionTreeClassifier(max_depth=3, random_state=42)
tree_honest.fit(X_honest, y)
honest_score = tree_honest.score(X_honest, y)
print(f"HONEST accuracy (no leakage): {honest_score:.3f}")
print()

# --- THE TRAP: add total_clicks itself, which directly defines the label ---
leaky_features = honest_features + ["total_clicks"]
X_leaky = feat[leaky_features].fillna(0)

tree_leaky = DecisionTreeClassifier(max_depth=3, random_state=42)
tree_leaky.fit(X_leaky, y)
leaky_score = tree_leaky.score(X_leaky, y)
print(f"LEAKY accuracy (total_clicks included -- label-derived!): {leaky_score:.3f}")
print()

print("The leak: total_clicks is literally what defines is_declining (bottom 30% by clicks).")
print("Including it lets the model 'cheat' by nearly reading the label directly,")
print("jumping the score toward-perfect. This is not a real signal -- it is leakage.")
print()
print(f"Removing the leaky feature and keeping the honest score: {honest_score:.3f}")

HONEST accuracy (no leakage): 0.844

LEAKY accuracy (total_clicks included -- label-derived!): 1.000

The leak: total_clicks is literally what defines is_declining (bottom 30% by clicks).
Including it lets the model 'cheat' by nearly reading the label directly,
jumping the score toward-perfect. This is not a real signal -- it is leakage.

Removing the leaky feature and keeping the honest score: 0.844


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [5]:
print("Named limitation of this slice:")
print()
print("Only 37% of rows (3.6M of 9.8M) actually have GSC data available")
print("(gsc_data_available IS TRUE) -- most clients in this warehouse don't have")
print("search console access connected, so my feature frame silently drops the")
print("majority of pages. This means my model only 'sees' and can only be useful")
print("for the subset of clients with GSC connected -- it says nothing about")
print("performance for clients without that access, and the 176,738-page feature")
print("frame is not representative of the full client base.")
print()
print("Also: this is a single month (March 2026) on the mid-panel -- seasonal")
print("effects specific to March won't be visible, and window overlaps at the")
print("edges of the month (first/last few days) have less trailing history than")
print("days in the middle.")

Named limitation of this slice:

Only 37% of rows (3.6M of 9.8M) actually have GSC data available
(gsc_data_available IS TRUE) -- most clients in this warehouse don't have
search console access connected, so my feature frame silently drops the
majority of pages. This means my model only 'sees' and can only be useful
for the subset of clients with GSC connected -- it says nothing about
performance for clients without that access, and the 176,738-page feature
frame is not representative of the full client base.

Also: this is a single month (March 2026) on the mid-panel -- seasonal
effects specific to March won't be visible, and window overlaps at the
edges of the month (first/last few days) have less trailing history than
days in the middle.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.